# Install Requirements

In [8]:
!pip install openpyxl pyarrow fastparquet

  Using cached pyarrow-21.0.0-cp311-cp311-win_amd64.whl.metadata (3.4 kB)
Using cached pyarrow-21.0.0-cp311-cp311-win_amd64.whl (26.2 MB)
   ---------------------------------------- 0.0/671.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/671.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/671.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/671.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/671.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/671.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/671.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/671.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/671.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/671.0 kB ? eta -:--:--
   --------------- ------------------------ 262.1/671.0 kB ? eta -:--:--
   --------------- ------------------------ 262.1/671.0 kB ? eta -:--:--
   ---


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Path Configuration

In [ ]:
from pathlib import Path
import pandas as pd
import json, re
from typing import Optional

IN_CSV = Path("../Dataset Creation/raw_dataset.csv")
OUT_CSV = "processed_dataset.csv"
OUT_PARQUET = "processed_dataset.parquet"
OUT_XLSX = "processed_dataset.xlsx"

print("Input:", IN_CSV)
print("Output CSV:", OUT_CSV)

Input: ..\Dataset Creation\raw_dataset.csv
Output CSV: processed_dataset.csv


# Helper Functions (Parsing and Regex)

In [ ]:
_re_digits = re.compile(r"([\d\.,]+)")
_re_generic_area = re.compile(r"([0-9]{1,4}(?:[.,][0-9]{1,3})?)\s*(?:m2|m²|m\s*2|m)", flags=re.I)
_re_bed = re.compile(r"(?:kamar\s*tidur|KT|kamar tidur|kt)\b[\s:\-]*([0-9]{1,2})", flags=re.I)
_re_bath = re.compile(r"(?:kamar\s*mandi|KM|kamar mandi|km)\b[\s:\-]*([0-9]{1,2})", flags=re.I)
_re_floor = re.compile(r"(?:lantai|lt\.?|floor|p_floor)[\s:]*([0-9]+)", flags=re.I)
_re_area_m = re.compile(r"(?:luas\s*(?:bangunan|bangunan|bang)|lb|LB|luas)[:\s\-]*([0-9\.,]+)\s*(?:m2|m²|m|m2)?", flags=re.I)

def to_float_from_number_string(s):
    if s is None:
        return None
    s = str(s).strip()
    s = re.sub(r"[^\d\.,\-]", "", s)
    s = s.replace(",", "")
    if s == "":
        return None
    try:
        return float(s)
    except:
        try:
            return float(s.split()[0])
        except:
            return None

def parse_main_info(main_info: Optional[str]):
    # contoh: "3 KT - 4 KM - 200 m2"
    if not main_info or not isinstance(main_info, str):
        return None, None, None
    bed = bath = area = None
    try:
        m = re.search(r"(\d+)\s*KT", main_info, flags=re.I)
        if m:
            bed = int(m.group(1))
        m = re.search(r"(\d+)\s*KM", main_info, flags=re.I)
        if m:
            bath = int(m.group(1))
        # area
        m = re.search(r"([0-9]{1,4}(?:[.,][0-9]{1,3})?)\s*(?:m2|m²|m)", main_info, flags=re.I)
        if m:
            area = float(m.group(1).replace(",", "."))
    except Exception:
        pass
    return bed, bath, area

def parse_raw_params(raw_params_str: Optional[str]):
    if not raw_params_str:
        return {}
    if isinstance(raw_params_str, dict):
        return raw_params_str
    try:
        d = json.loads(raw_params_str)
        if isinstance(d, dict):
            return d
    except:
        # try to fix single quotes -> double quotes
        try:
            d = json.loads(raw_params_str.replace("'", '"'))
            if isinstance(d, dict):
                return d
        except:
            # fallback regex key:value
            kvs = {}
            for m in re.finditer(r'"?([a-zA-Z0-9_\-]+)"?\s*:\s*"([^"]+)"', raw_params_str):
                kvs[m.group(1)] = m.group(2)
            return kvs
    return {}

def extract_from_description(desc: Optional[str]):
    out = {}
    if not desc or not isinstance(desc, str):
        return out
    # bedrooms
    m = _re_bed.search(desc)
    if m:
        try: out['bedrooms'] = int(m.group(1))
        except: pass
    # bathrooms
    m = _re_bath.search(desc)
    if m:
        try: out['bathrooms'] = int(m.group(1))
        except: pass
    # building area: try explicit "Luas Bangunan" then generic
    m = re.search(r"luas\s*(?:bangunan|bang|lb)[:\s\-]*([0-9\.,]+)", desc, flags=re.I)
    if m:
        out['building_area_m2'] = to_float_from_number_string(m.group(1))
    else:
        m = _re_generic_area.search(desc)
        if m:
            out['building_area_m2'] = to_float_from_number_string(m.group(1))
    # floor
    m = _re_floor.search(desc)
    if m:
        try: out['floor'] = int(m.group(1))
        except: pass
    else:
        m = re.search(r"(\d+)\s*lantai", desc, flags=re.I)
        if m:
            try: out['floor'] = int(m.group(1))
            except: pass
    # p_alamat (Alamat atau Lokasi)
    m = re.search(r"(?:Lokasi|Alamat|Location)[:\s\-]*([^\n\r]+)", desc, flags=re.I)
    if m:
        candidate = m.group(1).strip()
        candidate = re.split(r"(Harga|Price|Rp|IDR)", candidate)[0].strip()
        if candidate:
            out['p_alamat'] = candidate
    # external url in description
    m = re.search(r"(https?://[^\s]+)", desc)
    if m:
        out['external_url_from_desc'] = m.group(1)
    return out

# Baca dan Tampilkan Summary Awal

In [ ]:
assert IN_CSV.exists(), f"{IN_CSV} tidak ditemukan. Jalankan merge_olx_jsons_fixed.py dulu."
df = pd.read_csv(IN_CSV, dtype=object)

# konversi beberapa kolom numeric agar .notnull() lebih informatif
num_cols = ["price_raw","building_area_m2","land_area_m2","bedrooms","bathrooms","floor","lat","lon"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print("Rows:", len(df))
print("\nNon-null counts (sebelum):")
print(df.notnull().sum().sort_values(ascending=False))
df.head(3)

Rows: 25384

Non-null counts (sebelum):
ad_id                  25384
created_at             25384
main_info              25384
first_image_url        25384
images_count           25384
status                 25384
user_id                25384
lon                    25384
building_area_m2       25384
lat                    25384
city_name              25384
province_name          25384
country_name           25384
raw_parameters         25384
source_file            25384
bathrooms              25384
land_area_m2           25384
bedrooms               25384
title                  25383
user_name              25381
description            25376
property_type          25348
sublocality_name       25285
price_raw              25125
price_display          25125
currency               25125
external_source_url    22001
floor                  10031
p_alamat                3156
dtype: int64


,ad_id,title,description,price_raw,price_display,currency,images_count,first_image_url,main_info,created_at,...,property_type,building_area_m2,land_area_m2,bedrooms,bathrooms,floor,external_source_url,p_alamat,raw_parameters,source_file
0,934358929,Rumah 2lt 10x15 private pool type 3KT Cluster ...,Rumah 2lt 10x15 148m private pool type 3KT Clu...,3.500000e+09,Rp 3.500.000.000,IDR,7,https://apollo.olx.co.id:443/v1/files/68d2d116...,3 KT - 4 KM - 200 m2,2025-10-09T11:35:56+07:00,...,rumah,200.0,148.0,3,4,NaN,https://www.lamudi.co.id/properti/41032-73-cc7...,NaN,"{""external_source_url"": ""https://www.lamudi.co...",olx_combined_bekasi-kab_g4000003.json
1,937074956,RUMAH jual di GRIYA HARAPAN PERMAI. Kawasan ha...,RUMAH jual di GRIYA HARAPAN PERMAI. Kawasan h...,1.450000e+09,Rp 1.450.000.000,IDR,20,https://apollo.olx.co.id:443/v1/files/68ddabd3...,4 KT - 2 KM - 140 m2,2025-10-02T05:31:52+07:00,...,rumah,140.0,180.0,4,2,NaN,https://www.lamudi.co.id/properti/41032-73-cc6...,NaN,"{""external_source_url"": ""https://www.lamudi.co...",olx_combined_bekasi-kab_g4000003.json
2,923761993,Samesta East Point Tipe 2 Bedroom,Promo Samesta East Point Oktober 2024\n- Rp2 J...,8.445000e+08,Rp 844.500.000,IDR,9,https://apollo.olx.co.id:443/v1/files/68db7176...,2 KT - 1 KM - 44 m2,2025-09-30T12:58:17+07:00,...,apartemen,44.0,0.0,2,1,0.0,https://www.lamudi.co.id/proyek-baru/sl-samest...,NaN,"{""external_source_label"": ""https://www.lamudi....",olx_combined_bekasi-kab_g4000003.json


# Lakukan Fallback Filling (Vectorized/Apply Approach)

In [14]:
# parse raw params into a column of dicts
df["_params_dict"] = df.get("raw_parameters", pd.Series([None]*len(df))).apply(parse_raw_params)

# parse main_info
parsed_main = df.get("main_info", pd.Series([None]*len(df))).apply(lambda x: parse_main_info(x) if pd.notnull(x) else (None,None,None))
df["_main_bed"] = parsed_main.apply(lambda t: t[0])
df["_main_bath"] = parsed_main.apply(lambda t: t[1])
df["_main_area"] = parsed_main.apply(lambda t: t[2])

# helper - function to attempt fill of a numeric column using multiple sources
def fill_numeric_col(df, colname, param_keys=None, main_source=None):
    # param_keys: list of keys to look up in _params_dict
    for idx, row in df[df[colname].isna()].iterrows():
        val = None
        params = row.get("_params_dict") or {}
        if param_keys:
            for k in param_keys:
                if k in params and params[k] not in (None, ""):
                    val = params[k]
                    break
        if val is None and main_source:
            val = row.get(main_source)
        if val is None:
            # try description heuristics
            desc = row.get("description")
            ext = extract_from_description(desc)
            val = ext.get(colname) if ext else None
        if val is not None and val != "":
            # coerce to numeric
            try:
                if colname in ("bedrooms","bathrooms","floor"):
                    df.at[idx, colname] = int(float(str(val)))
                else:
                    df.at[idx, colname] = float(str(val).replace(",", "").strip())
            except:
                try:
                    df.at[idx, colname] = float(re.sub(r"[^\d\.]", "", str(val)))
                except:
                    pass

# run fills
fill_numeric_col(df, "bedrooms", param_keys=["p_bedroom","p_bedrooms","bedroom","p_bedrooms"], main_source="_main_bed")
fill_numeric_col(df, "bathrooms", param_keys=["p_bathroom","bathroom","p_bathrooms"], main_source="_main_bath")
fill_numeric_col(df, "building_area_m2", param_keys=["p_sqr_building","p_sqr_bangunan","building_area","luas_bangunan"], main_source="_main_area")
fill_numeric_col(df, "land_area_m2", param_keys=["p_sqr_land","land_area","luas_tanah"], main_source=None)
fill_numeric_col(df, "floor", param_keys=["p_floor","floor"], main_source=None)

# p_alamat and external_source_url fallback
def fill_address_and_external(df):
    for idx, row in df[df["p_alamat"].isna()].iterrows():
        params = row.get("_params_dict") or {}
        val = None
        for k in ("p_alamat","alamat","address"):
            if k in params and params[k] not in (None, ""):
                val = params[k]; break
        if val is None:
            ext = extract_from_description(row.get("description"))
            val = ext.get("p_alamat")
        if val:
            df.at[idx, "p_alamat"] = str(val).strip()
    # external_source_url
    for idx, row in df[df["external_source_url"].isna()].iterrows():
        params = row.get("_params_dict") or {}
        val = None
        for k in ("external_source_url","external_source_label","external_url","external_link"):
            if k in params and params[k] not in (None, ""):
                val = params[k]; break
        if val is None:
            ext = extract_from_description(row.get("description"))
            val = ext.get("external_url_from_desc")
        if val:
            df.at[idx, "external_source_url"] = val

fill_address_and_external(df)

# Summary Akhir dan Sample

In [ ]:
before_counts = None
after_counts = df.notnull().sum().sort_values(ascending=False)
print("Non-null counts (setelah):")
print(after_counts)

# lihat beberapa contoh baris dengan perubahan
display_cols = ["ad_id","title","price_raw","bedrooms","bathrooms","building_area_m2","land_area_m2","floor","p_alamat","external_source_url","lat","lon","city_name"]
available = [c for c in display_cols if c in df.columns]
df[available].head(8)

Non-null counts (setelah):
ad_id                  25384
first_image_url        25384
main_info              25384
images_count           25384
city_name              25384
country_name           25384
status                 25384
user_id                25384
created_at             25384
province_name          25384
raw_parameters         25384
bedrooms               25384
bathrooms              25384
land_area_m2           25384
building_area_m2       25384
lon                    25384
lat                    25384
_main_bath             25384
_main_area             25384
_main_bed              25384
_params_dict           25384
source_file            25384
title                  25383
user_name              25381
description            25376
property_type          25348
sublocality_name       25285
currency               25125
price_raw              25125
price_display          25125
external_source_url    22001
floor                  18877
p_alamat               12300
dtype: int64


,ad_id,title,price_raw,bedrooms,bathrooms,building_area_m2,land_area_m2,floor,p_alamat,external_source_url,lat,lon,city_name
0,934358929,Rumah 2lt 10x15 private pool type 3KT Cluster ...,3.500000e+09,3,4,200.0,148.0,10.0,NaN,https://www.lamudi.co.id/properti/41032-73-cc7...,-6.213,106.947,Bekasi Kota
1,937074956,RUMAH jual di GRIYA HARAPAN PERMAI. Kawasan ha...,1.450000e+09,4,2,140.0,180.0,5.0,NaN,https://www.lamudi.co.id/properti/41032-73-cc6...,-6.192,106.976,Bekasi Kota
2,923761993,Samesta East Point Tipe 2 Bedroom,8.445000e+08,2,1,44.0,0.0,0.0,strategis serta terintegrasi dengan transporta...,https://www.lamudi.co.id/proyek-baru/sl-samest...,-6.182,106.947,Jakarta Timur
3,932901804,Dijual Murah Apartemen WGP Full Furnished Di K...,4.600000e+08,1,1,40.0,0.0,6.0,NaN,https://www.lamudi.co.id/properti/41032-73-abd...,-6.158,106.918,Jakarta Utara
4,939957960,Rumah Siap Huni Bebas Banjir Lokasi Strategis ...,5.000000e+08,2,1,45.0,50.0,1.0,NaN,https://www.lamudi.co.id/properti/41032-73-d3c...,-6.230,106.960,Bekasi Kota
5,938232794,Rumah hook murah istimewa di pulo gebang perma...,1.795000e+09,4,3,300.0,184.0,2.0,NaN,https://www.lamudi.co.id/properti/41032-73-e41...,-6.196,106.955,Jakarta Timur
6,938509884,Dijual Apartement Gading Nias Residence Kelapa...,2.450000e+08,2,1,45.0,0.0,22.0,NaN,https://www.lamudi.co.id/properti/41032-73-b42...,-6.158,106.918,Jakarta Utara
7,938511300,Dijual cepat rumah dalam cluster di Metland Me...,1.800000e+09,3,2,124.0,96.0,96.0,NaN,https://www.lamudi.co.id/properti/41032-73-284...,-6.213,106.947,Bekasi Kota


# Simpan Hasil ke CSV/Parquet/Excel

In [ ]:
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
try:
    df.to_parquet(OUT_PARQUET, index=False)
except Exception as e:
    print("Parquet save issue:", e)
# Excel (xlsx)
try:
    # pastikan numeric cols sebagai numeric agar Excel rapi
    for c in ["price_raw","building_area_m2","land_area_m2","bedrooms","bathrooms","floor","lat","lon"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df.to_excel(OUT_XLSX, index=False)
    print("Saved Excel:", OUT_XLSX)
except Exception as e:
    print("Excel save error:", e)

print("Saved CSV:", OUT_CSV)
print("Saved Parquet:", OUT_PARQUET)

Parquet save issue: A type extension with name pandas.period already defined
Excel save error: Rumah ini berlokasi di kawasan Puri Gading, perumahan yang tertata rapi dengan suasana tenang dan nyaman. Lokasinya strategis, hanya sekitar 5 menit dari exit tol JORR, serta memiliki akses cepat menuju Cikunir, Jatiwarna, Bekasi, dan Jakarta Timur. 

Spesifikasi Rumah:
Luas Tanah: 240 m²
Luas Bangunan: 360 m²
Kamar Tidur: 4+1
Kamar Mandi: 4+1
5 Unit AC
Ruang Makan Terpisah
Carport: 2 Mobil
Garasi: 2 Mobil
Listrik: 2200 Watt
Sertifikat: SHM

Rumah dibangun dari nol dengan struktur yang kokoh. Rumah dalam kondisi terawat dan siap huni. Cocok untuk keluarga yang mencari rumah besar yang nyaman dengan akses mudah ke pusat kota. cannot be used in worksheets.
Saved CSV: processed_dataset.csv
Saved Parquet: processed_dataset.parquet


# Simpan Contoh Debug Kecil dan Statistik Kualitas Ekstraksi

In [17]:
# Simpan 200 sample acak untuk verifikasi manual jika ingin
debug_sample = df.sample(n=min(200, len(df)), random_state=42)
debug_sample.to_json("debug_manual_check_200.json", orient="records", force_ascii=False)
print("Saved manual-check sample:", "debug_manual_check_200.json")
# Tampilkan berapa banyak yang terisi untuk beberapa kolom penting
cols_check = ["bedrooms","bathrooms","building_area_m2","land_area_m2","floor","p_alamat","external_source_url","lat","lon"]
print(df[cols_check].notnull().sum().to_dict())

Saved manual-check sample: debug_manual_check_200.json
{'bedrooms': 25384, 'bathrooms': 25384, 'building_area_m2': 25384, 'land_area_m2': 25384, 'floor': 18877, 'p_alamat': 12300, 'external_source_url': 22001, 'lat': 25384, 'lon': 25384}
